# 🔗 PREP_03 — Integrasi Dataset X1 + X2 + Y (v3)
## Tesis: Pemodelan Prediksi Volatilitas Return Saham Sektor Energi LQ45
**Ilda Nurida — 6032251068 | Pembimbing: Dr. Ir. Ahmad Saikhu, S.Si, MT.**

### 🔧 Perbaikan tertanam (4 patch)
1. **Target Y TIDAK di-clip** → menghapus "dataran datar" palsu (mis. PGAS) saat volatilitas test > maksimum train.
2. **17 fitur konsisten** (13 teknikal + 4 sentimen, sinkron PREP_01 v2 & PREP_02 v2) → XGBoost & LSTM memakai set fitur identik.
3. **Sentimen forward-fill dari PREP_02 v2** (bukan flat netral) — hari tanpa berita mewarisi sentimen hari bursa terakhir, dengan `Ada_Berita=0` sebagai penanda.
4. **Scaler di-fit hanya pada train** (anti data-leakage), split kronologis 80/20.

### [v3] Revisi reviewer
5. **19 fitur** (15 teknikal termasuk `R_Exit` & `GARCH_Forecast` dari PREP_01 v4 + 4 sentimen). Guard otomatis menolak metadata lama.
6. **Fitur X pada data uji TIDAK di-clip** ke [0,1] — nilai > 1 berarti volatilitas uji di atas maksimum data latih; informasi ini dibutuhkan model (terutama LSTM) saat lonjakan volatilitas. Tidak ada kebocoran data karena scaler tetap di-fit hanya pada data latih.

### 📤 Output (kompatibel dgn MODEL_xgboost_FINAL / MODEL_lstm_FINAL / EVALUASI_perbandingan)
`dataset_train_{emiten}.csv`, `dataset_test_{emiten}.csv`, `scaler_params_{emiten}.csv`

In [1]:
import os, datetime
from pathlib import Path
import pandas as pd

# ════════════════════════════════════════════════════════════════════
#  PATH KANONIK + IO UTILITAS — IDENTIK DI SEMUA NOTEBOOK
#  (otomatis disisipkan oleh proses audit path)
# ════════════════════════════════════════════════════════════════════
ROOT_HARDCODED = Path(r'E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
                      r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026')

def cari_root():
    p = Path(os.getcwd())
    for _ in range(8):
        if (p / '03_dataset').exists():
            return p
        p = p.parent
    if ROOT_HARDCODED.exists():
        return ROOT_HARDCODED
    raise FileNotFoundError('ROOT proyek (folder berisi 03_dataset/) tidak ditemukan.')

ROOT = cari_root()
DIR_RAW       = ROOT / '03_dataset' / 'raw'
DIR_PROCESSED = ROOT / '03_dataset' / 'processed'
DIR_FINAL     = ROOT / '03_dataset' / 'final'
DIR_MODEL     = ROOT / '04_model'
DIR_TABEL     = ROOT / '06_dokumen_tesis' / 'tabel'
for _d in (DIR_RAW, DIR_PROCESSED, DIR_FINAL, DIR_TABEL):
    _d.mkdir(parents=True, exist_ok=True)

# Nama file kanonik
F_HARGA    = 'harga_saham_raw.csv'         # DIR_RAW
F_TEKNIKAL = 'technical_features.csv'      # DIR_PROCESSED (satu-satunya lokasi)
F_SENTIMEN = 'skor_sentimen_harian.csv'    # DIR_PROCESSED (nama tunggal)
F_GABUNGAN = 'dataset_gabungan_X1X2Y.csv'  # DIR_FINAL

def muat_csv(path, **kw):
    """LOAD + log sumber absolut. Error eksplisit bila tak ada (TANPA fallback diam-diam)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File TIDAK ADA: {path.resolve()}\n'
                                f'   -> Tahap sebelumnya belum menyimpan ke lokasi kanonik ini.')
    df = pd.read_csv(path, **kw)
    print(f'LOAD  : {path.resolve()}  ({len(df):,} baris x {df.shape[1]} kolom)')
    return df

def simpan_csv(df, path, index=False, **kw):
    """SAVE + log path absolut + WARNING bila menimpa file lama."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        mt = datetime.datetime.fromtimestamp(path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        print(f'OVERWRITE: {path.resolve()}  (file lama {mt} ditimpa)')
    df.to_csv(path, index=index, encoding='utf-8-sig', **kw)
    print(f'SAVE  : {path.resolve()}  ({len(df):,} baris)')

print(f'ROOT : {ROOT.resolve()}')
print(f'  RAW={DIR_RAW.name}/  PROCESSED={DIR_PROCESSED.name}/  FINAL={DIR_FINAL.name}/')


ROOT : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026
  RAW=raw/  PROCESSED=processed/  FINAL=final/


## ⚙️ CELL 1 — Konfigurasi

In [2]:
import os
import json as _json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# 1. PENENTUAN ROOT DIREKTORI KANONIK PROYEK TESIS
ROOT = Path(os.getcwd())
for _ in range(6):
    if (ROOT / '03_dataset').exists():
        break
    ROOT = ROOT.parent
if not (ROOT / '03_dataset').exists():
    ROOT = Path(
        r'e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
        r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026'
    )

# 2. STRUKTURISASI MULTI-LEVEL FOLDER PIPELINE
DIR_RAW = ROOT / '03_dataset' / 'raw'
DIR_PROCESSED = ROOT / '03_dataset' / 'processed'
DIR_FINAL = ROOT / '03_dataset' / 'final'
DIR_TABEL = ROOT / '06_dokumen_tesis' / 'tabel'

for d in (DIR_FINAL, DIR_TABEL):
    d.mkdir(parents=True, exist_ok=True)

# 3. METADATA EKSPERIMEN DATA PANEL
URUTAN_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']
SPLIT_RATIO = 0.80
NETRAL_SENTIMEN = 0.0  # skala Sentiment_Mean PREP_02 v2 adalah [-1,1], netral = 0 (BUKAN 0.5)

# ══════════════════════════════════════════════════════════════════
# 4. OPERASIONALISASI VARIABEL X1/X2/Y — REVISI AUDIT (FIX BUG #1,#2,#3)
# ══════════════════════════════════════════════════════════════════
# FIX BUG #1 & #2: KOLOM_X1 & KOLOM_Y dibaca OTOMATIS dari
# technical_features_meta.json (diekspor PREP_01 v2 CELL 10) — bukan
# hardcode manual yang gampang drift setiap kali fitur X1 direvisi.
# Ini juga otomatis memastikan KOLOM_Y = 'Volatilitas_Target' (forecast
# h+1), bukan 'Volatilitas' (nowcast) seperti versi lama.
PATH_META_X1 = DIR_PROCESSED / 'technical_features_meta.json'
if PATH_META_X1.exists():
    with open(PATH_META_X1, 'r', encoding='utf-8') as _f:
        _meta = _json.load(_f)
    KOLOM_X1 = _meta['kolom_x1']
    KOLOM_Y = _meta['kolom_y']
    print(f'  ℹ️  KOLOM_X1 & KOLOM_Y dimuat dari metadata PREP_01 v2: {PATH_META_X1.name}')
else:
    # Fallback kalau PREP_01 belum pernah dijalankan dengan versi v2 —
    # HARUS identik dengan KOLOM_X1_FINAL di PREP_01_technical_indicators_v2
    KOLOM_X1 = [
        'Log_Return', 'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'Vol_MA5', 'Vol_STD5',
        'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'RSI14',
        'Stoch_K', 'Stoch_D', 'Volume_ZScore',
        'R_Exit', 'GARCH_Forecast',            # [v3] fitur hybrid GARCH (PREP_01 v4)
    ]
    KOLOM_Y = 'Volatilitas_Target'
    print(f'  ⚠️  {PATH_META_X1.name} tidak ditemukan → pakai KOLOM_X1 fallback '
          f'hardcoded. Jalankan PREP_01 v2 dulu untuk metadata resmi.')

# FIX BUG #3: KOLOM_X2 disesuaikan PERSIS dengan output SEL M/N PREP_02 v2
# (Sentiment_Mean, Sentiment_Dispersion, News_Count, Ada_Berita) — bukan
# 8 fitur Lag/MA/STD lama yang tidak pernah benar-benar dihasilkan PREP_02.
KOLOM_X2 = [
    'Sentiment_Mean',
    'Sentiment_Dispersion',
    'News_Count',
    'Ada_Berita',
]  # 4 Fitur Sentimen — HARUS identik dengan kolom skor_sentimen_harian.csv

# [v3] GUARD — pastikan metadata berasal dari PREP_01 v4 (bukan versi lama)
_wajib_v4 = ['R_Exit', 'GARCH_Forecast', 'Vol_Lag1']
assert all(c in KOLOM_X1 for c in _wajib_v4) and 'Vol_Lag0' not in KOLOM_X1 and len(KOLOM_X1) == 15, (
    f"❌ Metadata X1 bukan versi PREP_01 v4: {KOLOM_X1}\n"
    f"   → Jalankan ulang PREP_01_technical_indicators_TESIS_FIX_v4 dulu (harus 15 fitur, tanpa Vol_Lag0).")

KOLOM_X = KOLOM_X1 + KOLOM_X2

# 5. KAMUS STANDARISASI STRUKTUR KOLOM ANTAR-PIPELINE (DATA QUALITY ASSURANCE)
# Catatan: alias MACD/BB/EMA lama DIHAPUS (fitur itu sudah tidak ada di
# pipeline v2). Alias Kode_Emiten/Tanggal dipertahankan sebagai jaring
# pengaman kalau ada file lama yang masih pakai penamaan itu.
PETA_VARIAN = {
    'RSI': 'RSI14',
    'RSI_14': 'RSI14',
    'Stoch_%K': 'Stoch_K',
    'Stoch_%D': 'Stoch_D',
    '%K': 'Stoch_K',
    '%D': 'Stoch_D',
    'Kode_Emiten': 'Emiten',
    'Tanggal': 'Date',
}

# 6. CETAK LAPORAN MONITORING INTEGRITAS KONFIGURASI
print('=' * 66)
print('  ⚙️  PREP_03 v2 — KONFIGURASI PROYEK TESIS BERHASIL DIKUNCI')
print('=' * 66)
print(f'  ROOT METODOLOGI : {ROOT}')
print(
    f'  TOTAL PREDIKTOR : {len(KOLOM_X)} Fitur (X1_Teknikal={len(KOLOM_X1)}, X2_Sentimen={len(KOLOM_X2)})'
)
print(f'  KOLOM_X1        : {KOLOM_X1}')
print(f'  KOLOM_X2        : {KOLOM_X2}')
print(f'  TARGET MODEL Y  : {KOLOM_Y} (Volatilitas Return Saham T+1 — forecast, bukan nowcast)')
print(
    f'  VALIDASI SPLIT  : Rasio Train={SPLIT_RATIO:.0%} | Nilai Netral Sentimen={NETRAL_SENTIMEN}'
)
print('=' * 66)


  ℹ️  KOLOM_X1 & KOLOM_Y dimuat dari metadata PREP_01 v2: technical_features_meta.json
  ⚙️  PREP_03 v2 — KONFIGURASI PROYEK TESIS BERHASIL DIKUNCI
  ROOT METODOLOGI : e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026
  TOTAL PREDIKTOR : 19 Fitur (X1_Teknikal=15, X2_Sentimen=4)
  KOLOM_X1        : ['Log_Return', 'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'Vol_MA5', 'Vol_STD5', 'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'RSI14', 'Stoch_K', 'Stoch_D', 'Volume_ZScore', 'R_Exit', 'GARCH_Forecast']
  KOLOM_X2        : ['Sentiment_Mean', 'Sentiment_Dispersion', 'News_Count', 'Ada_Berita']
  TARGET MODEL Y  : Volatilitas_Target (Volatilitas Return Saham T+1 — forecast, bukan nowcast)
  VALIDASI SPLIT  : Rasio Train=80% | Nilai Netral Sentimen=0.0


## 🔧 CELL 2 — Fungsi Bantu

In [3]:
def normalisasi_nama(df, peta=PETA_VARIAN):
    """Ubah nama kolom varian -> kanonik, hanya bila kanonik belum ada."""
    rename = {a:k for a,k in peta.items() if a in df.columns and k not in df.columns}
    return df.rename(columns=rename)

print('✅ Fungsi bantu siap.')


✅ Fungsi bantu siap.


## 📂 CELL 3 — Load Data Teknikal (X1 + Y)

In [4]:
# ==============================================================================
# CELL — PEMUATAN DATA TEKNIKAL KANONIK (DATA QUALITY ASSURANCE X1)
# ==============================================================================

# Fungsi penjaminan mutu untuk menyamakan varian nama kolom bursa ke standar tesis
def normalisasi_nama(dataframe, pemetaan=PETA_VARIAN):
    if dataframe is None or dataframe.empty:
        return dataframe
    # Mengubah nama kolom berdasarkan kamus besar PETA_VARIAN di Cell 1
    kolom_baru = {k: v for k, v in pemetaan.items() if k in dataframe.columns}
    return dataframe.rename(columns=kolom_baru)

print("-" * 75)
print(" ⚙️ MEMULAI PEMUATAN DATA TEKNIKAL DARI LOKASI KANONIK (DIR_PROCESSED)")
print("-" * 75)

# 1. LOAD teknikal dari SATU lokasi kanonik (processed/), TANPA fallback
df_tek = normalisasi_nama(muat_csv(DIR_PROCESSED / F_TEKNIKAL))

# 2. Pembersihan kronologis dan penormalan stempel tanggal murni (bebas jam)
df_tek['Date'] = pd.to_datetime(df_tek['Date'], errors='coerce')
df_tek = df_tek.dropna(subset=['Date'])
df_tek['Date'] = df_tek['Date'].dt.normalize()

# 3. Filter ketat data panel hanya untuk 7 emiten energi jangkar penelitian
df_tek = df_tek[df_tek['Emiten'].isin(URUTAN_EMITEN)].copy()

# 4. Asersi dan Pengecekan Ketersediaan Kolom Varial (X1 + Y)
tek_hilang = [c for c in KOLOM_X1 + [KOLOM_Y] if c not in df_tek.columns]

# 5. Cetak Laporan Integritas Pipeline Data Teknikal
if not tek_hilang:
    str_baris_tek = f"{len(df_tek):,}".replace(",", ".")
    print(f'  ✅ SUKSES: {len(KOLOM_X1)} fitur teknikal + target Y lengkap ({str_baris_tek} baris).')
else:
    print(f'  ⚠️ PERINGATAN: Kolom prediktor/target hilang: {tek_hilang}')
    print(f'     Silakan periksa kembali kecocokan string di PETA_VARIAN (Cell 1).')
print("-" * 75)


---------------------------------------------------------------------------
 ⚙️ MEMULAI PEMUATAN DATA TEKNIKAL DARI LOKASI KANONIK (DIR_PROCESSED)
---------------------------------------------------------------------------
LOAD  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\processed\technical_features.csv  (7,494 baris x 27 kolom)
  ✅ SUKSES: 15 fitur teknikal + target Y lengkap (7.494 baris).
---------------------------------------------------------------------------


## 📂 CELL 4 — Load Data Sentimen Harian (X2)

In [5]:
# ==============================================================================
# CELL — PEMUATAN DATA SENTIMEN HARIAN (REVISI AUDIT — FIX BUG #3)
# ==============================================================================

print("-" * 75)
print(" ⚙️ MEMULAI PEMUATAN DATA SENTIMEN DARI LOKASI KANONIK (DIR_PROCESSED)")
print("-" * 75)

# 1. LOAD sentimen dari SATU nama kanonik (skor_sentimen_harian.csv)
df_sen = muat_csv(DIR_PROCESSED / F_SENTIMEN, low_memory=False)

# FIX AUDIT: alias HANYA untuk variasi Emiten/Date yang genuinely mungkin
# beda antar sumber lama. TIDAK ADA lagi fallback diam-diam untuk kolom
# skor sentimen — kalau nama kolom X2 tidak sama persis dengan output
# PREP_02 v2, notebook harus GAGAL EKSPLISIT (sesuai filosofi muat_csv()
# di CELL 0), bukan diam-diam mengganti sinyal asli dengan konstanta.
RENAME_SEN_KUNCI = {
    'tanggal': 'Date', 'Tanggal': 'Date',
    'kode_emiten': 'Emiten', 'Kode_Emiten': 'Emiten',
}
df_sen = df_sen.rename(columns={k: v for k, v in RENAME_SEN_KUNCI.items() if k in df_sen.columns})

# 2. Pembersihan & normalisasi komponen tanggal
df_sen['Date'] = pd.to_datetime(df_sen['Date'], errors='coerce')
df_sen = df_sen.dropna(subset=['Date'])
df_sen['Date'] = df_sen['Date'].dt.normalize()

# 3. Filter ketat berdasarkan 7 emiten kandidat energi jangkar penelitian
df_sen = df_sen[df_sen['Emiten'].isin(URUTAN_EMITEN)].copy()

# 4. FIX AUDIT (BUG #3): validasi KETAT — GAGAL EKSPLISIT kalau kolom X2
# tidak ditemukan, alih-alih diam-diam mengisi dengan nilai konstan
# (bug versi lama: seluruh sinyal sentimen asli terbuang jadi 0.5/1 flat).
kolom_x2_hilang = [c for c in KOLOM_X2 if c not in df_sen.columns]
if kolom_x2_hilang:
    raise ValueError(
        f"❌ Kolom X2 tidak ditemukan di {F_SENTIMEN}: {kolom_x2_hilang}\n"
        f"   Kolom tersedia: {df_sen.columns.tolist()}\n"
        f"   → Jalankan ulang PREP_02 v2 (SEL M/N) supaya kolom "
        f"'Sentiment_Mean/Sentiment_Dispersion/News_Count/Ada_Berita' tersimpan."
    )

# 5. Agregasi (jaga-jaga bila ada duplikat Emiten+Date dari run PREP_02 berulang)
df_sen = (
    df_sen.groupby(['Emiten', 'Date'], as_index=False)[KOLOM_X2]
    .mean()
)
# Ada_Berita & News_Count seharusnya integer, mean() di atas bisa membuatnya
# float kalau ada duplikat — kembalikan ke bentuk semula
df_sen['Ada_Berita'] = (df_sen['Ada_Berita'] > 0).astype(int)
df_sen['News_Count'] = df_sen['News_Count'].round().astype(int)

# 6. Laporan Status Kesiapan Fitur Sentimen
str_baris_sen = f"{df_sen.shape[0]:,}".replace(",", ".")
t_min_sen = df_sen["Date"].min().date()
t_max_sen = df_sen["Date"].max().date()

print(f'  ✅ SUKSES: {len(KOLOM_X2)} fitur sentimen ({str_baris_sen} baris x {df_sen.shape[1]} kolom).')
print(f'     Kolom X2: {KOLOM_X2}')
print(f'     Rentang Histori: {t_min_sen} s/d {t_max_sen}')
print("-" * 75)


---------------------------------------------------------------------------
 ⚙️ MEMULAI PEMUATAN DATA SENTIMEN DARI LOKASI KANONIK (DIR_PROCESSED)
---------------------------------------------------------------------------
LOAD  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\processed\skor_sentimen_harian.csv  (7,518 baris x 6 kolom)
  ✅ SUKSES: 4 fitur sentimen (7.518 baris x 6 kolom).
     Kolom X2: ['Sentiment_Mean', 'Sentiment_Dispersion', 'News_Count', 'Ada_Berita']
     Rentang Histori: 2022-01-03 s/d 2026-06-30
---------------------------------------------------------------------------


## 🔗 CELL 5 — Merge + Pengisian Netral + 8 Fitur Sentimen (Patch 2 & 3)

In [6]:
# ==============================================================================
# CELL — LOAD TECHNICAL FEATURES + MERGE SENTIMENT (REVISI AUDIT)
# ==============================================================================

print("-" * 80)
print("⚙️ LOAD TECHNICAL FEATURES + MERGE SENTIMENT")
print("-" * 80)

# ============================================================
# 1. LOAD DATA TEKNIKAL TERBARU
# ============================================================
df_tek = muat_csv(DIR_PROCESSED / F_TEKNIKAL, low_memory=False)

rename_tek = {'Tanggal': 'Date', 'Kode_Emiten': 'Emiten'}
df_tek.rename(
    columns={k: v for k, v in rename_tek.items() if k in df_tek.columns},
    inplace=True
)

df_tek['Date'] = pd.to_datetime(df_tek['Date'], errors='coerce')
df_tek = df_tek.dropna(subset=['Date']).copy()
df_tek['Date'] = df_tek['Date'].dt.normalize()

# ============================================================
# 2. VALIDASI FEATURE WAJIB — FIX AUDIT: 'Volatilitas_Target' (bukan
#    'Volatilitas_Return' yang sudah tidak ada di PREP_01 v2)
# ============================================================
kolom_vol_wajib = [
    'Volatilitas', 'Volatilitas_Target',
    'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'Vol_MA5', 'Vol_STD5'
]

kolom_hilang = [c for c in kolom_vol_wajib if c not in df_tek.columns]

if len(kolom_hilang) > 0:
    raise ValueError(
        f"❌ Technical features masih versi lama.\n"
        f"Kolom hilang: {kolom_hilang}\n"
        f"Silakan rerun PREP_01 v2 lalu overwrite technical_features.csv"
    )

print("✅ Semua feature volatility tersedia (termasuk Volatilitas_Target — forecast h+1)")

# ============================================================
# 3. HAPUS DUPLIKAT
# ============================================================
df_tek = (
    df_tek
    .drop_duplicates(['Emiten', 'Date'], keep='last')
    .sort_values(['Emiten', 'Date'])
    .reset_index(drop=True)
)

print(f"Technical rows: {len(df_tek):,}")

# ============================================================
# 4. MERGE DENGAN SENTIMEN
# FIX AUDIT: cukup merge 4 kolom X2 dari PREP_02 v2 langsung — TIDAK
# perlu lagi rekayasa manual Lag1/Lag2/MA3/MA20/STD20 dari 'Skor_Sentimen'
# (kolom itu sudah tidak ada; PREP_02 v2 sudah menyediakan Sentiment_Mean
# yang forward-filled + Sentiment_Dispersion secara langsung).
# ============================================================
df = pd.merge(
    df_tek,
    df_sen[['Emiten', 'Date'] + KOLOM_X2],
    on=['Emiten', 'Date'],
    how='left'
)

df = (
    df
    .drop_duplicates(['Emiten', 'Date'], keep='last')
    .sort_values(['Emiten', 'Date'])
    .reset_index(drop=True)
)

# ============================================================
# 5. IMPUTASI JIKA MASIH ADA TANGGAL YANG TIDAK KETEMU DI X2
# (seharusnya 0 baris kalau PREP_02 v2 pakai kalender technical_features.csv
# — ini murni jaring pengaman, bukan jalur utama)
# ============================================================
n_missing_x2 = df[KOLOM_X2].isna().any(axis=1).sum()
if n_missing_x2 > 0:
    print(f"  ⚠️  {n_missing_x2} baris tidak ketemu pasangan tanggal di X2 "
          f"→ diisi netral (Sentiment_Mean=0.0, Dispersion=0.0, News_Count=0, Ada_Berita=0)")
df['Sentiment_Mean'] = df['Sentiment_Mean'].fillna(NETRAL_SENTIMEN)
df['Sentiment_Dispersion'] = df['Sentiment_Dispersion'].fillna(0.0)
df['News_Count'] = df['News_Count'].fillna(0).astype(int)
df['Ada_Berita'] = df['Ada_Berita'].fillna(0).astype(int)

# ============================================================
# 6. REPORT
# ============================================================
print(f"Final merged rows: {len(df):,}")
print()
print("Kolom final:")
print(df.columns.tolist())
print("-" * 80)


--------------------------------------------------------------------------------
⚙️ LOAD TECHNICAL FEATURES + MERGE SENTIMENT
--------------------------------------------------------------------------------
LOAD  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\processed\technical_features.csv  (7,494 baris x 27 kolom)
✅ Semua feature volatility tersedia (termasuk Volatilitas_Target — forecast h+1)
Technical rows: 7,494
Final merged rows: 7,494

Kolom final:
['Date', 'Emiten', 'Open', 'Close', 'High', 'Low', 'Volume', 'Log_Return', 'Volatilitas', 'Volatilitas_Target', 'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'R_Exit', 'Vol_MA5', 'Vol_STD5', 'MA5', 'MA10', 'MA20', 'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'Stoch_K', 'Stoch_D', 'RSI14', 'Volume_ZScore', 'GARCH_Forecast', 'Sentiment_Mean', 'Sentiment_Dispersion', 'News_Count', 'Ada_Berita']
--------------------------------------------------------------------

## 📋 CELL 6 — Finalisasi Kolom & Buang Warm-up NaN

In [7]:
# ==============================================================================
# VALIDASI AKHIR, PEMBERSIHAN BARIS WARM-UP (NaN), DAN PENYIMPANAN DATASET
# ==============================================================================

# 1. Validasi Integritas Struktur Kolom (Data Quality Assurance)
sentimen_hilang = [c for c in KOLOM_X2 if c not in df.columns]
assert not sentimen_hilang, f'Fitur sentimen hilang: {sentimen_hilang}'
assert KOLOM_Y in df.columns, f'Target {KOLOM_Y} tidak ada di data teknikal!'

# 2. Pemotongan Kolom Sesuai Konfigurasi Tesis (KOLOM_X1 + KOLOM_X2 + Y)
KOLOM_PAKAI = [c for c in ['Date', 'Emiten'] + KOLOM_X + [KOLOM_Y] if c in df.columns]
df = df[KOLOM_PAKAI].copy()

# 3. Pembersihan Baris Kosong Akibat Efek Warm-Up Indikator (Lag, MA, MACD, BB)
n0 = len(df)
# Menghapus baris yang bernilai NaN pada semua fitur prediktor (X) dan target (Y)
df = df.dropna(subset=[c for c in KOLOM_X + [KOLOM_Y] if c in df.columns]).reset_index(drop=True)

# [FIXED] Menampilkan laporan reduksi baris dengan f-string terpadu format Indonesia (titik)
log_baris = f"  Baris: {n0:,.0f} → {len(df):,.0f} (buang {n0-len(df):,.0f} baris warm-up NaN)"
print(log_baris.replace(',', '.'))

# 4. Pelaporan Status Kelengkapan Variabel Sebelum Modeling
n_fitur = len([c for c in KOLOM_X if c in df.columns])
print(f'  ✅ Total fitur final: {n_fitur} / {len(KOLOM_X)}')
if n_fitur != len(KOLOM_X):
    print(f'  ⚠️  Kurang dari {len(KOLOM_X)} → cek kolom hilang sebelum lanjut.')

# 5. Konversi Tanggal dan Penyimpanan Ekspor Akhir ke DIR_FINAL
ds = df.copy()
# Memastikan kolom Date diubah menjadi string format ISO murni YYYY-MM-DD agar aman dibuka di Excel/CSV
ds['Date'] = pd.to_datetime(ds['Date']).dt.strftime('%Y-%m-%d')

# Pastikan variabel nama file F_GABUNGAN tersedia di lokal system, jika belum otomatis pakai default string
if 'F_GABUNGAN' not in locals():
    F_GABUNGAN = 'dataset_gabungan_X1X2Y.csv'

simpan_csv(ds, DIR_FINAL / F_GABUNGAN)
print(f'  💾 {F_GABUNGAN} tersimpan dengan sukses di {DIR_FINAL.name}/')


  Baris: 7.494 → 7.494 (buang 0 baris warm-up NaN)
  ✅ Total fitur final: 19 / 19
OVERWRITE: E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_gabungan_X1X2Y.csv  (file lama 2026-08-06 14:09 ditimpa)
SAVE  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_gabungan_X1X2Y.csv  (7,494 baris)
  💾 dataset_gabungan_X1X2Y.csv tersimpan dengan sukses di final/


## 🔢 CELL 7 — Split 80/20 Kronologis + MinMax (fit train) — Y TIDAK di-clip (Patch 1 & 4)

In [8]:
# ==============================================================================
# PEMBAGIAN DATA PANEL (TRAIN-TEST SPLIT) DAN NORMALISASI MINMAX KUSTOM - FIXED
# ==============================================================================

dict_train, dict_test, dict_scaler, dict_split = {}, {}, {}, {}
KOLOM_NORM = [c for c in KOLOM_X + [KOLOM_Y] if c in df.columns]

print('='*66)
print('  🔢 SPLIT + NORMALISASI (Fit pada training data saja)')
print('='*66)

for emiten in URUTAN_EMITEN:
    # Filter data per emiten dan pastikan urutan waktunya kronologis
    df_e = df[df['Emiten'] == emiten].sort_values('Date').reset_index(drop=True)
    if df_e.empty:
        print(f'  ⚠️  {emiten}: data kosong, skip')
        continue

    # [SOLUSI KRITIS] Ratakan kolom jika terdeteksi MultiIndex akibat sisa proses merge/yfinance
    if isinstance(df_e.columns, pd.MultiIndex):
        df_e.columns = [c[0] if isinstance(c, tuple) else c for c in df_e.columns]

    # Split rasio menggunakan urutan waktu kronologis (tanpa acak/shuffle)
    n_train = int(len(df_e) * SPLIT_RATIO)
    df_tr = df_e.iloc[:n_train].copy()
    df_te = df_e.iloc[n_train:].copy()

    scaler_params = {}
    
    # Proses Normalisasi MinMax manual Fit-on-Train untuk mencegah Data Leakage
    for col in KOLOM_NORM:
        # Mengambil nilai minimum & maksimum skalar tunggal terlepas dari tipe data array
        val_min = df_tr[col].min()
        val_max = df_tr[col].max()
        
        # Ekstraksi nilai skalar jika objek masih membungkus struktur series
        cmin = float(val_min.iloc[0]) if hasattr(val_min, 'iloc') else float(val_min)
        cmax = float(val_max.iloc[0]) if hasattr(val_max, 'iloc') else float(val_max)
        
        scaler_params[col] = {'min': cmin, 'max': cmax}
        
        denom = cmax - cmin
        if denom == 0:
            df_tr[col] = 0.0
            df_te[col] = 0.0
        else:
            # Transformasi linier pada data latih dan data uji
            df_tr[col] = (df_tr[col] - cmin) / denom
            df_te[col] = (df_te[col] - cmin) / denom
            
            # [v3] X dan Y pada data uji TIDAK di-clip. Nilai > 1 / < 0 berarti di luar
            # rentang data latih (mis. lonjakan volatilitas); informasi ini dipertahankan.

    # Simpan parameter scaler, dataset hasil transformasi, dan meta split data
    dict_train[emiten] = df_tr
    dict_test[emiten] = df_te
    dict_scaler[emiten] = scaler_params
    
    # Pengamanan: Mengubah string tanggal ke objek Date secara eksplisit sebelum mengambil nilai .date()
    t_tr_min = pd.to_datetime(df_tr['Date'].min()).date()
    t_tr_max = pd.to_datetime(df_tr['Date'].max()).date()
    t_te_min = pd.to_datetime(df_te['Date'].min()).date()
    t_te_max = pd.to_datetime(df_te['Date'].max()).date()
    
    # Atasi ekstraksi tanggal jika bernilai objek array/series akibat multiindex
    t_tr_min = t_tr_min.iloc[0] if hasattr(t_tr_min, 'iloc') else t_tr_min
    t_tr_max = t_tr_max.iloc[0] if hasattr(t_tr_max, 'iloc') else t_tr_max
    t_te_min = t_te_min.iloc[0] if hasattr(t_te_min, 'iloc') else t_te_min
    t_te_max = t_te_max.iloc[0] if hasattr(t_te_max, 'iloc') else t_te_max
    
    dict_split[emiten] = {
        'Emiten': emiten, 'n_total': len(df_e), 'n_train': n_train, 'n_test': len(df_te),
        'train_start': t_tr_min, 'train_end': t_tr_max,
        'test_start': t_te_min, 'test_end': t_te_max
    }
    
    # Laporan visual pengawasan target nilai maksimum volatilitas di masa pengujian
    ymax = df_te[KOLOM_Y].max()
    ymax_val = float(ymax.iloc[0]) if hasattr(ymax, 'iloc') else float(ymax)
    tag = '  (>1 → ada lonjakan vol uji di atas tren latih; TIDAK terpotong)' if ymax_val > 1 else ''
    
    # Format pemisah ribuan standar Indonesia (titik)
    str_train = f"{n_train:,}".replace(",", ".")
    str_test = f"{len(df_te):,}".replace(",", ".")
    print(f'  ✅ {emiten:5s}: train={str_train:5s} test={str_test:5s} | Y_test_max(scaled)={ymax_val:.2f}{tag}')

print('='*66)


  🔢 SPLIT + NORMALISASI (Fit pada training data saja)
  ✅ ADRO : train=859   test=215   | Y_test_max(scaled)=0.40
  ✅ ADMR : train=840   test=210   | Y_test_max(scaled)=0.85
  ✅ ITMG : train=859   test=215   | Y_test_max(scaled)=1.02  (>1 → ada lonjakan vol uji di atas tren latih; TIDAK terpotong)
  ✅ PTBA : train=859   test=215   | Y_test_max(scaled)=0.50
  ✅ MEDC : train=859   test=215   | Y_test_max(scaled)=0.80
  ✅ AKRA : train=859   test=215   | Y_test_max(scaled)=0.48
  ✅ PGAS : train=859   test=215   | Y_test_max(scaled)=1.34  (>1 → ada lonjakan vol uji di atas tren latih; TIDAK terpotong)


## 💾 CELL 8 — Simpan Dataset Final per Emiten

In [9]:
print('='*66); print('  SIMPAN dataset_train/test + scaler_params'); print('='*66)
for emiten in URUTAN_EMITEN:
    if emiten not in dict_train: continue
    tr = dict_train[emiten].copy(); te = dict_test[emiten].copy()
    tr['Date'] = pd.to_datetime(tr['Date']).dt.strftime('%Y-%m-%d')
    te['Date'] = pd.to_datetime(te['Date']).dt.strftime('%Y-%m-%d')
    simpan_csv(tr, DIR_FINAL / f'dataset_train_{emiten}.csv')
    simpan_csv(te, DIR_FINAL / f'dataset_test_{emiten}.csv')
    sc = pd.DataFrame(dict_scaler[emiten]).T[['min','max']]; sc.index.name = 'kolom'
    simpan_csv(sc, DIR_FINAL / f'scaler_params_{emiten}.csv', index=True)
print(f'\n  Semua dataset final di: {DIR_FINAL.resolve()}')


  SIMPAN dataset_train/test + scaler_params
OVERWRITE: E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_train_ADRO.csv  (file lama 2026-08-06 14:09 ditimpa)
SAVE  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_train_ADRO.csv  (859 baris)
OVERWRITE: E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_test_ADRO.csv  (file lama 2026-08-06 14:09 ditimpa)
SAVE  : E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\dataset_test_ADRO.csv  (215 baris)
OVERWRITE: E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\03_dataset\final\scaler_par

## 📋 CELL 9 — Tabel Split & Ringkasan (→ Tabel 4.X)

In [10]:
df_split = pd.DataFrame([dict_split[e] for e in URUTAN_EMITEN if e in dict_split])
df_split.index = range(1, len(df_split)+1); df_split.index.name='No'
print(df_split.to_string())
df_split.to_csv(DIR_TABEL / 'tabel_split_train_test.csv', encoding='utf-8-sig')

print('\n' + '='*66)
print('  ✅ PREP_03 FINAL SELESAI')
print('='*66)
print(f'  Perbaikan aktif: [1] Y tak di-clip  [2] {len(KOLOM_X)} fitur (X1={len(KOLOM_X1)}+X2={len(KOLOM_X2)})  [3] sentimen sinkron PREP_02 v2  [4] fit train-only')
print('  → Lanjut: MODEL_xgboost_FINAL.ipynb → MODEL_lstm_FINAL.ipynb → EVALUASI_perbandingan.ipynb')
print('='*66)

   Emiten  n_total  n_train  n_test train_start   train_end  test_start    test_end
No                                                                                 
1    ADRO     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30
2    ADMR     1050      840     210  2022-02-07  2025-08-14  2025-08-15  2026-06-30
3    ITMG     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30
4    PTBA     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30
5    MEDC     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30
6    AKRA     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30
7    PGAS     1074      859     215  2022-01-03  2025-08-07  2025-08-08  2026-06-30

  ✅ PREP_03 FINAL SELESAI
  Perbaikan aktif: [1] Y tak di-clip  [2] 19 fitur (X1=15+X2=4)  [3] sentimen sinkron PREP_02 v2  [4] fit train-only
  → Lanjut: MODEL_xgboost_FINAL.ipynb → MODEL_lstm_FINAL.ipynb → EVALUASI_perbandingan.ipynb


In [11]:
# ============================================================
# CELL 10 [v3] — VALIDASI AKHIR (wajib lolos sebelum modeling)
# ============================================================
print("LEN KOLOM_X1:", len(KOLOM_X1), "| LEN KOLOM_X:", len(KOLOM_X))
print(KOLOM_X1)
for emiten in URUTAN_EMITEN:
    tr, te = dict_train[emiten], dict_test[emiten]
    assert list(tr.columns) == list(te.columns)
    assert all(c in te.columns for c in KOLOM_X + [KOLOM_Y]), f'{emiten}: kolom hilang'
    assert tr[KOLOM_X + [KOLOM_Y]].isna().sum().sum() == 0 and te[KOLOM_X + [KOLOM_Y]].isna().sum().sum() == 0, f'{emiten}: ada NaN'
    assert pd.to_datetime(tr['Date']).max() < pd.to_datetime(te['Date']).min(), f'{emiten}: split tidak kronologis'
    print(f"  [OK] {emiten}: {len(KOLOM_X)} fitur | train {len(tr)} | test {len(te)} | "
          f"GARCH_Forecast test max (scaled) = {te['GARCH_Forecast'].max():.2f}")
assert len(KOLOM_X) == 19, f'Jumlah fitur {len(KOLOM_X)}, seharusnya 19'
print('\n[OK] Dataset siap untuk MODEL_xgboost & MODEL_lstm (19 fitur)')

LEN KOLOM_X1: 15 | LEN KOLOM_X: 19
['Log_Return', 'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'Vol_MA5', 'Vol_STD5', 'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'RSI14', 'Stoch_K', 'Stoch_D', 'Volume_ZScore', 'R_Exit', 'GARCH_Forecast']
  [OK] ADRO: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 0.40
  [OK] ADMR: 19 fitur | train 840 | test 210 | GARCH_Forecast test max (scaled) = 0.85
  [OK] ITMG: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 1.02
  [OK] PTBA: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 0.50
  [OK] MEDC: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 0.82
  [OK] AKRA: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 0.46
  [OK] PGAS: 19 fitur | train 859 | test 215 | GARCH_Forecast test max (scaled) = 1.34

[OK] Dataset siap untuk MODEL_xgboost & MODEL_lstm (19 fitur)
